In [15]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [5]:
torch.manual_seed(11037)

In [6]:
class ProjectRedMLP(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.hidden = nn.Sequential(    nn.Linear(5, 32),
                                        nn.ReLU(),
                                        nn.Linear(32, 16),
                                        nn.ReLU()         )

        self.center = nn.Linear(16, 1)

        self.width = nn.Sequential(     nn.Linear(16, 1),
                                        nn.Softplus()     )

    def forward(self, x):

        x = self.hidden(x)
        center = self.center(x)
        width = self.width(x) 

        return torch.cat((center, width), dim=1)

Softplus is because it is an activation that is linear at higher values so it doesn't blow up with vals.

In [7]:
model = ProjectRedMLP()
print(model)

ProjectRedMLP(
  (hidden): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (center): Linear(in_features=16, out_features=1, bias=True)
  (width): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
    (1): Softplus(beta=1.0, threshold=20.0)
  )
)


In [9]:
df = pd.read_csv("project_red_center_width.csv")
df.head()

,user_id,last_problem_id,user_rating,last_problem_rating,mean_solve_time_minutes,time_std_minutes,problems_solved,center,width,lower_bound,upper_bound,synthetic
0,SYN_000000,2109:C3,2263.0,2600,31.77,4.34,64,2417.25,298.93,2267.78,2566.72,True
1,SYN_000001,606:B,1414.0,1600,31.28,9.94,84,1555.10,568.89,1270.65,1839.54,True
2,SYN_000002,913:D,1833.0,1800,29.39,11.96,79,1844.16,426.22,1631.05,2057.27,True
3,SYN_000003,1419:D2,1290.0,1500,41.23,17.98,142,1347.95,588.71,1053.59,1642.31,True
4,SYN_000004,1428:H,3385.0,3500,31.50,17.31,306,3396.72,226.51,3283.46,3509.97,True


In [10]:
features = ["user_rating", "last_problem_rating", "mean_solve_time_minutes", "time_std_minutes", "problems_solved"]

targets = ["center", "width"]

X = df[features]
Y = df[targets]

In [11]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=11037)

In [13]:
scaler_x = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test =  scaler_x.fit_transform(X_test)

In [16]:
joblib.dump(scaler_x, "scaler_x.pkl")

['scaler_x.pkl']